In [0]:
# siver 테이블 가져오기
silver_df = spark.read.table("training.sjh.amazon_delivery_silver")

In [0]:
silver_df.show()

+-------------+---------+------------+--------------+---------------+-------------+--------------+----------+----------+-------------------+----------+-------+----------+-------------+-------------+-----------+
|     Order_ID|Agent_Age|Agent_Rating|Store_Latitude|Store_Longitude|Drop_Latitude|Drop_Longitude|Order_Date|Order_Time|        Pickup_Time|   Weather|Traffic|   Vehicle|         Area|Delivery_Time|   Category|
+-------------+---------+------------+--------------+---------------+-------------+--------------+----------+----------+-------------------+----------+-------+----------+-------------+-------------+-----------+
|ialx566343618|       37|         4.9|     22.745049|      75.892471|    22.765049|     75.912471|2022-03-19|  11:30:00|2026-08-30 11:45:00|     Sunny|   High|motorcycle|        Urban|          120|   Clothing|
|akqg208421122|       34|         4.5|     12.913041|      77.683237|    13.043041|     77.813237|2022-03-25|  19:45:00|2026-08-30 19:50:00|    Stormy|    J

In [0]:
# 배송 소요 시간을 30분 단위로 구간화하여 평점 집계

import pyspark.sql.functions as F

# 2. 배송 소요 시간 구간화 및 KPI 집계
gold_df = (
    silver_df
    # Delivery_Time(분) 기준 구간(Bucket) 생성
    .withColumn(
        "delivery_time_bucket",
        F.when(F.col("Delivery_Time") <= 30, "01_Under_30m")
         .when((F.col("Delivery_Time") > 30) & (F.col("Delivery_Time") <= 60), "02_30~60m")
         .when((F.col("Delivery_Time") > 60) & (F.col("Delivery_Time") <= 90), "03_60~90m")
         .when((F.col("Delivery_Time") > 90) & (F.col("Delivery_Time") <= 120), "04_90~120m")
         .when((F.col("Delivery_Time") > 120) & (F.col("Delivery_Time") <= 150), "05_120~150m")
         .when((F.col("Delivery_Time") > 150) & (F.col("Delivery_Time") <= 180), "06_150~180m")
         .otherwise("07_Over_180m")
    )
    # 구간별 그룹화 및 집계
    .groupBy("delivery_time_bucket")
    .agg(
        F.count("Order_ID").alias("total_orders"),
        F.round(F.avg("Agent_Rating"), 2).alias("avg_agent_rating"),
        F.min("Delivery_Time").alias("min_delivery_time_min"),
        F.max("Delivery_Time").alias("max_delivery_time_min")
    )
    .orderBy("delivery_time_bucket")
)

In [0]:
gold_df.show()

+--------------------+------------+----------------+---------------------+---------------------+
|delivery_time_bucket|total_orders|avg_agent_rating|min_delivery_time_min|max_delivery_time_min|
+--------------------+------------+----------------+---------------------+---------------------+
|        01_Under_30m|        1849|            4.73|                   10|                   30|
|           02_30~60m|        2866|            4.66|                   31|                   60|
|           03_60~90m|        7546|            4.74|                   65|                   90|
|          04_90~120m|        9013|            4.72|                   95|                  120|
|         05_120~150m|       10152|            4.72|                  125|                  150|
|         06_150~180m|        5773|            4.38|                  155|                  180|
|        07_Over_180m|        6395|            4.44|                  185|                  270|
+--------------------+--------

In [0]:
# gold_df 저장하기
gold_df.write.mode("overwrite").saveAsTable("training.sjh.amazon_delivery_gold")

In [0]:
%sql
SELECT * FROM training.sjh.amazon_delivery_gold

delivery_time_bucket,total_orders,avg_agent_rating,min_delivery_time_min,max_delivery_time_min
01_Under_30m,1849,4.73,10,30
02_30~60m,2866,4.66,31,60
03_60~90m,7546,4.74,65,90
04_90~120m,9013,4.72,95,120
05_120~150m,10152,4.72,125,150
06_150~180m,5773,4.38,155,180
07_Over_180m,6395,4.44,185,270
